# 03 - Lakeflow Connect: managed ingestion

Lakeflow Connect brings data **into** the lakehouse with managed, incremental connectors. You configure *what* to ingest and Databricks runs it as a serverless pipeline that lands **streaming tables** in Unity Catalog. You then transform them with SDP, SQL or PySpark (see `pipeline/`).

## Connector families
| Family | Examples | How it works |
| --- | --- | --- |
| **SaaS applications** | Salesforce, Workday (Reports / HCM), ServiceNow, HubSpot, Jira, Google Analytics, Google Ads, Meta Ads | UC **connection** (OAuth/credentials) -> **ingestion pipeline** calling the source API incrementally |
| **Databases (CDC)** | SQL Server, PostgreSQL, MySQL, Oracle | UC connection -> **ingestion gateway** (always-on, reads change logs/binlogs, stages to a volume) -> **ingestion pipeline** applies changes. MySQL also has an integrated (gateway-less) CDC mode |
| **Unstructured / file sources** | SharePoint, Google Drive | Managed file ingestion into tables |
| **Standard connectors** | Auto Loader (cloud storage / volumes), Kafka, Kinesis, Event Hubs, Pub/Sub | You write the code (SDP or Structured Streaming), see notebook 02 and `pipeline/` |

## Architecture
`Source -> UC Connection -> [Ingestion Gateway: DBs only] -> Ingestion Pipeline (serverless) -> Streaming tables (SCD1 / SCD2) -> SDP transformations -> Gold`

## Why use it (use cases)
* Replace custom API scripts or 3rd-party ELT tools for common SaaS/DB sources
* Incremental sync & CDC out of the box (cursor columns, change tracking, deletes)
* Handles API rate limits, pagination, retries and **schema evolution**
* Governed by Unity Catalog (connections, lineage, permissions); orchestrated with Lakeflow Jobs
* SCD Type 1 or Type 2 history at ingestion, column selection, row filters (Beta, SaaS)

## Limitations / things to plan for
* **Coverage**: only supported connectors & objects; check per-connector docs for unsupported objects/fields and preview status
* **Source-side quotas**: SaaS API limits (e.g. Salesforce daily API calls) bound how often you can sync
* **Database prerequisites**: CDC / change tracking enabled on the source, network connectivity, replication user; the **gateway runs continuously** (cost) and must keep up before logs are purged
* **Transformations** are limited (column selection, row filters, SCD type); business logic belongs downstream in SDP
* Some options are **API-only** (column selection, multi-destination, destination rename); renaming makes the pipeline non-editable in the UI
* Connector-specific caps (e.g. Workday Reports: 2 GB / 1M records per report, unique primary keys required, max 250 reports per pipeline)

## Step 1: Find (or create) a UC connection
Connections hold the source credentials and are governed by Unity Catalog. Create them in **Catalog > External data > Connections > Create connection**, or start from **Data ingestion** in the sidebar (UI wizard). The cell below lists connections you can see, grouped by type.

In [0]:
from pyspark.sql import functions as F

INGESTION_TYPES = ["SALESFORCE", "WORKDAY_RAAS", "SERVICENOW", "GOOGLE_ANALYTICS", "SQLSERVER", "POSTGRESQL",
                   "MYSQL", "ORACLE", "GOOGLE_DRIVE", "SHAREPOINT", "HUBSPOT", "JIRA"]
conns = spark.sql("SHOW CONNECTIONS").filter(F.col("type").isin(INGESTION_TYPES))
display(conns.groupBy("type").count().orderBy(F.desc("count")))
display(conns.select("name", "type", "owner", "comment").orderBy("type", "name"))

## Step 2: Create ingestion pipelines as code
The UI wizard is the fastest path; the API / Declarative Automation Bundles give you repeatable, reviewable definitions (and some API-only options). The helpers use the Databricks SDK, which is pre-authenticated in notebooks.

`CREATE_PIPELINES = False` keeps this demo safe: the specs are only printed. Fill in a real connection name and set it to `True` to actually create them.

In [0]:
import json
from databricks.sdk import WorkspaceClient

dbutils.widgets.text("catalog", "main", "Catalog")
dbutils.widgets.text("schema", "de_demo", "Schema")
DEST_CATALOG = dbutils.widgets.get("catalog")
DEST_SCHEMA = dbutils.widgets.get("schema")

w = WorkspaceClient()
CREATE_PIPELINES = False

def create_pipeline(spec: dict):
    print(json.dumps(spec, indent=2))
    if not CREATE_PIPELINES:
        print("\n(dry mode) not created")
        return None
    resp = w.api_client.do("POST", "/api/2.0/pipelines", body=spec)
    print("created pipeline:", resp["pipeline_id"])
    return resp["pipeline_id"]

def start_pipeline(pipeline_id: str, full_refresh: bool = False):
    return w.api_client.do("POST", f"/api/2.0/pipelines/{pipeline_id}/updates",
                           body={"full_refresh": full_refresh, "cause": "API_CALL"})

### Example A: Salesforce (SaaS)
Salesforce objects use `source_schema: "objects"`. `table_configuration` shows SCD Type 2 history and column exclusion.

In [0]:
salesforce_spec = {
    "name": "de_demo_salesforce_ingest",
    "ingestion_definition": {
        "connection_name": "<your_salesforce_connection>",
        "objects": [
            {"table": {"source_schema": "objects", "source_table": "Account",
                       "destination_catalog": DEST_CATALOG, "destination_schema": DEST_SCHEMA,
                       "table_configuration": {"scd_type": "SCD_TYPE_2"}}},
            {"table": {"source_schema": "objects", "source_table": "Opportunity",
                       "destination_catalog": DEST_CATALOG, "destination_schema": DEST_SCHEMA,
                       "table_configuration": {"exclude_columns": ["Description"]}}},
        ],
    },
}
sf_id = create_pipeline(salesforce_spec)

### Example B: PostgreSQL / SQL Server (database CDC)
Two objects: an **ingestion gateway** (continuous, captures changes into a staging volume) and an **ingestion pipeline** (applies them to streaming tables, typically scheduled with a Job). Note the gateway uses `connection_id`, not the name. You can ingest specific tables or a whole schema.

In [0]:
DB_CONNECTION_NAME = "<your_postgres_connection>"

connection_id = "<connection-id>"
if CREATE_PIPELINES:
    connection_id = w.connections.get(DB_CONNECTION_NAME).connection_id

gateway_spec = {
    "name": "de_demo_pg_gateway",
    "catalog": DEST_CATALOG,
    "schema": DEST_SCHEMA,
    "gateway_definition": {
        "connection_id": connection_id,
        "gateway_storage_catalog": DEST_CATALOG,
        "gateway_storage_schema": DEST_SCHEMA,
        "gateway_storage_name": "de_demo_pg_gateway",
    },
}
gateway_id = create_pipeline(gateway_spec) or "<gateway-pipeline-id>"

db_pipeline_spec = {
    "name": "de_demo_pg_ingest",
    "catalog": DEST_CATALOG,
    "schema": DEST_SCHEMA,
    "ingestion_definition": {
        "ingestion_gateway_id": gateway_id,
        "objects": [
            # a single table, with a custom destination name
            {"table": {"source_schema": "public", "source_table": "orders",
                       "destination_catalog": DEST_CATALOG, "destination_schema": DEST_SCHEMA,
                       "destination_table": "pg_orders"}},
            # or a whole schema (current and future tables)
            {"schema": {"source_schema": "sales",
                        "destination_catalog": DEST_CATALOG, "destination_schema": DEST_SCHEMA}},
        ],
    },
}
db_id = create_pipeline(db_pipeline_spec)

## Step 3: Transform downstream with SDP + orchestrate
Ingested tables are regular UC streaming tables, so the SDP pipeline can read them exactly like the Auto Loader bronze tables in this demo, e.g. `spark.readStream.table("<catalog>.<schema>.Account")` for a streaming silver table, or `spark.read.table(...)` for a gold materialized view. For SCD2 tables use the `__START_AT` / `__END_AT` columns (current rows: `__END_AT IS NULL`).

Typical production setup: one **Lakeflow Job** with tasks `ingestion pipeline -> SDP transformation pipeline -> dashboard refresh / alerts`, triggered on a schedule (the DB gateway runs continuously alongside).

## When *not* to use Lakeflow Connect
* Files in cloud storage / volumes -> **Auto Loader** (notebook 02 / SDP bronze)
* Event streams -> **Kafka / Kinesis / Event Hubs** readers in SDP or Structured Streaming
* Query data in place without copying -> **Lakehouse Federation**
* An unsupported source -> custom Python data source, partner ELT tools, or Lakeflow Connect community/partner connectors